In [ ]:
import os, sys
os.chdir(os.path.expanduser('~/QIAO0042/models/acv/facemask/'))
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())

CWD: /scratch-share/QIAO0042/models/acv/facemask


In [3]:
import os, time, logging
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pathlib import Path
from PIL import Image as PILImage
from contextlib import contextmanager

from unet import UNet, UNetV2
from losses import FocalDiceLoss, CEDiceLoss
from metrics import confusion_matrix, f1_macro_from_cm
from palette import NUM_CLASSES
from dataset import FaceParsingDataset
from split_utils import list_images, make_split
from augment import make_face_aug
from palette import rgb_to_label

logging.getLogger('torch._inductor').setLevel(logging.WARNING)
logging.getLogger('torch._dynamo').setLevel(logging.WARNING)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16  = device.type == 'cuda' and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
torch.backends.cudnn.benchmark = True
print(f'Device: {device}  AMP: {amp_dtype}')

Device: cuda  AMP: torch.bfloat16


In [6]:
# -----------------------------------------------------------------------
# Shared data preparation — used by all ablation runs
# -----------------------------------------------------------------------
IMG_DIR  = '/tmp/facemask/images'
MASK_DIR = '/tmp/facemask/masks'

# Copy to /tmp if not already there
import shutil
for src, dst in [('train/images', IMG_DIR), ('train/masks', MASK_DIR)]:
    if not Path(dst).exists():
        shutil.copytree(Path(src).resolve(), dst)
        print(f'  copied {src} -> {dst}')
    else:
        print(f'  already exists: {dst}')

SEED      = 42
VAL_RATIO = 0.1
EPOCHS    = 50   # reduced for ablation speed; increase to 100 for final numbers
BATCH     = 16
LR        = 2e-2
WARMUP    = 5
AUX_W     = 0.4

all_files = list_images(IMG_DIR)
train_files, val_files = make_split(all_files, val_ratio=VAL_RATIO, seed=SEED)
print(f'Split: {len(train_files)} train / {len(val_files)} val')

# Class weights over training set
pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
mask_lookup  = {p.stem: p for p in Path(MASK_DIR).iterdir()
                if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}}
for fn in train_files:
    labels = rgb_to_label(np.array(PILImage.open(mask_lookup[Path(fn).stem]).convert('RGB'), dtype=np.uint8))
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((labels == c).sum())

freq             = pixel_counts / pixel_counts.sum()
median_freq      = float(np.median(freq[freq > 0]))
class_weights_np = np.where(freq > 0, np.sqrt(median_freq / freq), 1.0)
CLASS_WEIGHTS    = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

aug_full  = make_face_aug(p_flip=0.5, p_geom=0.7, p_color=0.7, p_blur=0.15)

# Flip-only augmentation (no geometric/colour transforms)
from augment import flip_with_label_swap
def aug_flip_only(img, mask):
    return flip_with_label_swap(img, mask, p=0.5)

print('Data preparation done.')

  already exists: /tmp/facemask/images
  already exists: /tmp/facemask/masks
Split: 900 train / 100 val
Data preparation done.


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/scratch-share/QIAO0042/models/acv/facemask/augment.py:166: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(


In [4]:
# -----------------------------------------------------------------------
# Ablation configurations — FORWARD (incremental) design
# Each entry ADDS exactly ONE component on top of the previous config.
# Last entry == full system.
# -----------------------------------------------------------------------
ABLATION_CONFIGS = [
    {
        'name'         : 'scratch',
        'desc'         : 'Baseline: plain UNet, CE+Dice, flip-only aug, no extras',
        'model'        : 'unet',
        'loss'         : 'ce_dice',
        'ohem_ratio'   : 1.0,
        'use_weights'  : False,
        'augmentation' : 'flip_only',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+arch',
        'desc'         : 'scratch +UNetV2 (attention + ASPP)',
        'model'        : 'unetv2',
        'loss'         : 'ce_dice',
        'ohem_ratio'   : 1.0,
        'use_weights'  : False,
        'augmentation' : 'flip_only',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+focal',
        'desc'         : '+arch +Focal-Dice loss',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 1.0,
        'use_weights'  : False,
        'augmentation' : 'flip_only',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+weights',
        'desc'         : '+focal +class weights',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 1.0,
        'use_weights'  : True,
        'augmentation' : 'flip_only',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+ohem',
        'desc'         : '+weights +OHEM (ratio=0.7)',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 0.7,
        'use_weights'  : True,
        'augmentation' : 'flip_only',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+aug',
        'desc'         : '+ohem +full augmentation',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 0.7,
        'use_weights'  : True,
        'augmentation' : 'full',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : False,
    },
    {
        'name'         : '+deepsup',
        'desc'         : '+aug +deep supervision',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 0.7,
        'use_weights'  : True,
        'augmentation' : 'full',
        'use_ema'      : False,
        'use_tta'      : False,
        'deep_sup'     : True,
    },
    {
        'name'         : '+ema',
        'desc'         : '+deepsup +EMA (decay=0.999)',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 0.7,
        'use_weights'  : True,
        'augmentation' : 'full',
        'use_ema'      : True,
        'use_tta'      : False,
        'deep_sup'     : True,
    },
    {
        'name'         : '+tta',
        'desc'         : 'Full system: +ema +TTA (horizontal flip)',
        'model'        : 'unetv2',
        'loss'         : 'focal_dice',
        'ohem_ratio'   : 0.7,
        'use_weights'  : True,
        'augmentation' : 'full',
        'use_ema'      : True,
        'use_tta'      : True,
        'deep_sup'     : True,
    },
]

print(f'{len(ABLATION_CONFIGS)} configurations defined (forward / incremental).')
for i, cfg in enumerate(ABLATION_CONFIGS):
    print(f"  [{i}] {cfg['name']:12s}  {cfg['desc']}")

9 configurations defined (forward / incremental).
  [0] scratch       Baseline: plain UNet, CE+Dice, flip-only aug, no extras
  [1] +arch         scratch +UNetV2 (attention + ASPP)
  [2] +focal        +arch +Focal-Dice loss
  [3] +weights      +focal +class weights
  [4] +ohem         +weights +OHEM (ratio=0.7)
  [5] +aug          +ohem +full augmentation
  [6] +deepsup      +aug +deep supervision
  [7] +ema          +deepsup +EMA (decay=0.999)
  [8] +tta          Full system: +ema +TTA (horizontal flip)


In [5]:
# -----------------------------------------------------------------------
# Generic experiment runner
# -----------------------------------------------------------------------
_FLIP_PAIRS = [(4, 5), (6, 7), (8, 9)]  # l/r eye, brow, ear


class EMAKeeper:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.num_updates = 0
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        d = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.data, alpha=1 - d)

    @contextmanager
    def applied(self, model):
        original = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])
        try:
            yield
        finally:
            for n, p in model.named_parameters():
                if p.requires_grad:
                    p.data.copy_(original[n])


def _downsample_mask(masks, size):
    return F.interpolate(masks.float().unsqueeze(1), size=size, mode='nearest').squeeze(1).long()


@torch.no_grad()
def validate(model, loader, use_tta, ema=None):
    ctx = ema.applied(model) if ema is not None else contextmanager(lambda: (yield))()
    model.eval()
    cm_total = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)
    with ctx:
        for imgs, masks, _ in loader:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=device.type == 'cuda'):
                logits = model(imgs)
                if use_tta:
                    logits_flip = model(imgs.flip(-1)).flip(-1)
                    for a, b in _FLIP_PAIRS:
                        logits_flip[:, [a, b]] = logits_flip[:, [b, a]]
                    logits = (logits + logits_flip) * 0.5
            pred = logits.argmax(dim=1)
            cm_total += confusion_matrix(pred.cpu(), masks.cpu(), num_classes=NUM_CLASSES)
    return f1_macro_from_cm(cm_total)


def run_experiment(cfg):
    print(f"\n{'='*60}")
    print(f"  {cfg['name']}:  {cfg['desc']}")
    print(f"{'='*60}")

    # --- augmentation ---
    aug = aug_full if cfg['augmentation'] == 'full' else aug_flip_only

    # --- datasets ---
    train_ds = FaceParsingDataset(IMG_DIR, MASK_DIR, file_list=train_files, augment=aug, cache=True)
    val_ds   = FaceParsingDataset(IMG_DIR, MASK_DIR, file_list=val_files,   augment=None, cache=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)
    val_loader   = DataLoader(val_ds,   batch_size=8,     shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

    # --- model ---
    if cfg['model'] == 'unetv2':
        model = UNetV2(num_classes=NUM_CLASSES, base=23, dropout=0.3,
                       deep_supervision=cfg['deep_sup']).to(device)
    else:
        model = UNet(num_classes=NUM_CLASSES, base=23).to(device)
    try:
        model = torch.compile(model, mode='default')
    except Exception:
        pass

    # --- loss ---
    cw = CLASS_WEIGHTS if cfg['use_weights'] else None
    if cfg['loss'] == 'focal_dice':
        criterion = FocalDiceLoss(NUM_CLASSES, dice_weight=0.85, gamma=2.0,
                                  class_weights=cw, ohem_ratio=cfg['ohem_ratio'])
    else:
        criterion = CEDiceLoss(NUM_CLASSES, dice_weight=0.7, class_weights=cw)

    # --- optimizer + scheduler ---
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, 0.1, 1.0, total_iters=WARMUP),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP, eta_min=1e-6),
        ],
        milestones=[WARMUP],
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(not use_bf16 and device.type == 'cuda'))
    ema    = EMAKeeper(model, decay=0.999) if cfg['use_ema'] else None

    best_f1 = -1.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        running = 0.0

        for imgs, masks, _ in train_loader:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=device.type == 'cuda'):
                outputs = model(imgs)
                if isinstance(outputs, tuple):
                    main, aux1, aux2 = outputs
                    loss = (criterion(main, masks)
                            + AUX_W * criterion(aux1, _downsample_mask(masks, aux1.shape[-2:]))
                            + AUX_W * criterion(aux2, _downsample_mask(masks, aux2.shape[-2:])))
                else:
                    loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            if ema is not None:
                ema.update(model)
            running += loss.item()

        scheduler.step()
        val_f1, _ = validate(model, val_loader, use_tta=cfg['use_tta'], ema=ema)
        if val_f1 > best_f1:
            best_f1 = val_f1

        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"  [{epoch:03d}/{EPOCHS}] loss={running/len(train_loader):.4f}  "
                  f"val_F1={val_f1:.4f}  best={best_f1:.4f}  ({time.time()-t0:.1f}s)")

    print(f"  -> Best val macro-F1: {best_f1:.4f}")
    return best_f1


print('Experiment runner ready.')

Experiment runner ready.


In [ ]:
# -----------------------------------------------------------------------
# Run all ablation experiments and persist results after each run.
# Results are saved to ablation_results.json + ablation_results.csv so
# they survive a notebook disconnect / kernel restart.
# -----------------------------------------------------------------------
import json, csv
from datetime import datetime

RESULTS_JSON = Path('ablation_results.json')
RESULTS_CSV  = Path('ablation_results.csv')

def _load_saved_results():
    """Load any previously completed runs so we can skip/resume."""
    if RESULTS_JSON.exists():
        with open(RESULTS_JSON) as f:
            return json.load(f)
    return {}

def _save_results(results_dict):
    """Overwrite both JSON and CSV with the full results dict."""
    # JSON
    payload = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'epochs'   : EPOCHS,
        'seed'     : SEED,
        'results'  : results_dict,
    }
    with open(RESULTS_JSON, 'w') as f:
        json.dump(payload, f, indent=2)

    # CSV
    names = [cfg['name'] for cfg in ABLATION_CONFIGS]
    with open(RESULTS_CSV, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'best_val_f1', 'description'])
        for cfg in ABLATION_CONFIGS:
            name = cfg['name']
            f1   = results_dict.get(name, '')
            writer.writerow([name, f1, cfg['desc']])

    print(f'  [saved] {RESULTS_JSON}  {RESULTS_CSV}')


# Load any prior results (allows resuming after a disconnect)
results = _load_saved_results()
if results:
    print(f'Loaded {len(results)} previously saved result(s): {list(results.keys())}')

for cfg in ABLATION_CONFIGS:
    name = cfg['name']
    if name in results:
        print(f'  Skipping {name} (already done, F1={results[name]:.4f})')
        continue
    results[name] = run_experiment(cfg)
    _save_results(results)   # persist immediately after each run

print('\nAll experiments complete.')
print(f'Results saved to: {RESULTS_JSON.resolve()}  and  {RESULTS_CSV.resolve()}')


  scratch:  Baseline: plain UNet, CE+Dice, flip-only aug, no extras
  → cached 900 decoded arrays in RAM
Loaded 900 samples from /tmp/facemask/images
  → cached 100 decoded arrays in RAM
Loaded 100 samples from /tmp/facemask/images


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  [010/50] loss=0.8046  val_F1=0.4205  best=0.4643  (7.5s)
  [020/50] loss=0.5436  val_F1=0.7188  best=0.7198  (7.6s)
  [030/50] loss=0.4070  val_F1=0.7495  best=0.7500  (7.7s)
  [040/50] loss=0.3261  val_F1=0.7619  best=0.7655  (7.6s)
  [050/50] loss=0.2753  val_F1=0.7646  best=0.7656  (7.7s)
  -> Best val macro-F1: 0.7656
  [saved] ablation_results.json  ablation_results.csv

  +arch:  scratch +UNetV2 (attention + ASPP)
  → cached 900 decoded arrays in RAM
Loaded 900 samples from /tmp/facemask/images
  → cached 100 decoded arrays in RAM
Loaded 100 samples from /tmp/facemask/images


/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


  [010/50] loss=0.7023  val_F1=0.5976  best=0.5976  (8.4s)
  [020/50] loss=0.4877  val_F1=0.7318  best=0.7318  (8.4s)
  [030/50] loss=0.3736  val_F1=0.7637  best=0.7637  (8.6s)
  [040/50] loss=0.2993  val_F1=0.7749  best=0.7749  (8.5s)
  [050/50] loss=0.2738  val_F1=0.7773  best=0.7774  (8.4s)
  -> Best val macro-F1: 0.7774
  [saved] ablation_results.json  ablation_results.csv

  +focal:  +arch +Focal-Dice loss
  → cached 900 decoded arrays in RAM
Loaded 900 samples from /tmp/facemask/images
  → cached 100 decoded arrays in RAM
Loaded 100 samples from /tmp/facemask/images
  [010/50] loss=0.5623  val_F1=0.6189  best=0.6251  (8.9s)
  [020/50] loss=0.4133  val_F1=0.7389  best=0.7389  (8.8s)
  [030/50] loss=0.3298  val_F1=0.7543  best=0.7644  (8.8s)
  [040/50] loss=0.2712  val_F1=0.7711  best=0.7783  (8.8s)
  [050/50] loss=0.2300  val_F1=0.7783  best=0.7793  (8.9s)
  -> Best val macro-F1: 0.7793
  [saved] ablation_results.json  ablation_results.csv

  +weights:  +focal +class weights
  → c

In [7]:
# -----------------------------------------------------------------------
# Results table — forward/incremental view
# Shows absolute F1 and delta vs. the previous step.
# Can also be run standalone by loading the saved JSON.
# -----------------------------------------------------------------------
import json
from pathlib import Path

# Allow running this cell independently after a disconnect
if 'results' not in dir() or not results:
    saved = json.load(open('ablation_results.json'))
    results = saved['results']
    print(f"Loaded results from file (timestamp: {saved['timestamp']})")

f1_values = [results.get(cfg['name'], float('nan')) for cfg in ABLATION_CONFIGS]

print(f"\n{'#':<3}  {'Name':<12}  {'Val F1':>8}  {'Δ prev':>8}  {'Δ scratch':>9}  Description")
print('-' * 95)
for i, cfg in enumerate(ABLATION_CONFIGS):
    name     = cfg['name']
    f1       = f1_values[i]
    d_prev   = f1 - f1_values[i - 1] if i > 0 else 0.0
    d_base   = f1 - f1_values[0]
    arrow    = '  ▲' if d_prev > 0.005 else ('  ▼' if d_prev < -0.005 else '  ~')
    base_tag = '' if i == 0 else f'{d_base:+.4f}'
    print(f"[{i}]  {name:<12}  {f1:>8.4f}  {d_prev:>+8.4f}{arrow}  {base_tag:>9}  {cfg['desc']}")

print()
print(f"Baseline  (scratch):  {f1_values[0]:.4f}")
print(f"Full system (+tta):   {f1_values[-1]:.4f}  "
      f"(+{f1_values[-1]-f1_values[0]:.4f} over baseline)")
print()
print(f"Epochs per run: {EPOCHS}  |  Val split: {int(VAL_RATIO*100)}%  |  Seed: {SEED}")
print(f"Note: Δ prev > 0 means adding that component helped.")


#    Name            Val F1    Δ prev  Δ scratch  Description
-----------------------------------------------------------------------------------------------
[0]  scratch         0.7656   +0.0000  ~             Baseline: plain UNet, CE+Dice, flip-only aug, no extras
[1]  +arch           0.7774   +0.0118  ▲    +0.0118  scratch +UNetV2 (attention + ASPP)
[2]  +focal          0.7793   +0.0019  ~    +0.0138  +arch +Focal-Dice loss
[3]  +weights        0.7866   +0.0073  ▲    +0.0211  +focal +class weights
[4]  +ohem           0.7841   -0.0026  ~    +0.0185  +weights +OHEM (ratio=0.7)
[5]  +aug            0.7832   -0.0008  ~    +0.0177  +ohem +full augmentation
[6]  +deepsup        0.7870   +0.0038  ~    +0.0214  +aug +deep supervision
[7]  +ema            0.7871   +0.0001  ~    +0.0215  +deepsup +EMA (decay=0.999)
[8]  +tta            0.7883   +0.0013  ~    +0.0228  Full system: +ema +TTA (horizontal flip)

Baseline  (scratch):  0.7656
Full system (+tta):   0.7883  (+0.0228 over baseline)
